## ulta top10 순위 및 모든리뷰데이터 2년치 추출코드

In [1]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import requests
import json
import time
import random
from datetime import datetime, timedelta

# ── 설정 ──────────────────────────────────────────────────────────
API_KEY          = "daa0f241-c242-4483-afb7-4449942d1a2b"
REVIEW_SAVE_FILE = "ulta_reviews_master.jsonl"
RANK_SAVE_FILE   = "ulta_rankings_current.jsonl"

# [날짜 설정] 현재 기준 2년 전 타임스탬프 계산 (밀리초 단위)
CUTOFF_DATE = datetime.now() - timedelta(days=365 * 2)
CUTOFF_TIMESTAMP = CUTOFF_DATE.timestamp() * 1000  # 밀리초로 변환

# ── 함수: 단일 상품 리뷰 수집 (최근 2년치 필터링) ───────────────────
def get_product_reviews_2years(product_id, product_name):
    final_reviews = []
    page_size   = 25
    paging_from = 0
    stop_collecting = False

    print(f"\n  🚀 [{product_name[:30]}] 리뷰 수집 (기준: {CUTOFF_DATE.strftime('%Y-%m-%d')} 이후)")

    while not stop_collecting:
        url = f"https://display.powerreviews.com/m/6406/l/en_US/product/{product_id}/reviews"
        params = {
            "paging.from" : paging_from,
            "paging.size" : page_size,
            "sort"        : "Newest",  # 최신순 정렬 필수
            "apikey"      : API_KEY,
        }
        try:
            response = requests.get(url, params=params, timeout=10)
            if response.status_code != 200: break

            data = response.json()
            results = data.get("results", [])

            if not results or not results[0].get("reviews"):
                break

            reviews = results[0].get("reviews", [])
            for rev in reviews:
                details = rev.get("details", {})
                metrics = rev.get("metrics", {})
                
                # 1. created_date 추출 (밀리초 단위)
                raw_ms = details.get("created_date")
                
                if raw_ms:
                    # 2. 날짜 비교: 2년 전 타임스탬프보다 작으면 수집 중단
                    if raw_ms < CUTOFF_TIMESTAMP:
                        stop_collecting = True
                        break
                
                # 3. 데이터 저장
                fmt_date = datetime.fromtimestamp(raw_ms / 1000.0).strftime("%Y-%m-%d %H:%M") if raw_ms else "N/A"
                final_reviews.append({
                    "product_id" : product_id,
                    "review_id"  : rev.get("review_id"),
                    "author"     : details.get("nickname"),
                    "rating"     : metrics.get("rating"),
                    "headline"   : details.get("headline"),
                    "comment"    : details.get("comments"),
                    "date"       : fmt_date,
                    "created_at" : raw_ms
                })

            if stop_collecting:
                print(f"    📍 2년 이전 리뷰 도달 ({fmt_date}) - 수집 종료")
                break

            print(f"    🔄 수집 중... (누적: {len(final_reviews)}개)", end="\r")
            paging_from += page_size
            time.sleep(random.uniform(0.3, 0.5))

        except Exception as e:
            print(f"\n    ❌ 에러 발생: {e}")
            break

    print(f"\n    ✨ {len(final_reviews)}개 수집 완료")
    return final_reviews


# ── 메인: Ulta 데이터 수집 ─────────────────────────────────────────
def fetch_ulta_data():
    options = uc.ChromeOptions()
    options.add_argument("--window-size=1920,1080")
    driver = uc.Chrome(options=options)

    try:
        print(f"📡 Ulta 접속 중 (기준일: {CUTOFF_DATE.strftime('%Y-%m-%d')})")
        driver.get("https://www.ulta.com/shop/skin-care/all?sort=best_sellers")
        time.sleep(random.uniform(7, 10))

        soup  = BeautifulSoup(driver.page_source, "html.parser")
        items = soup.select("li.ProductListingResults__productCard")

        rank_data_list    = []
        all_review_master = []
        rank_count        = 1

        for item in items:
            if rank_count > 10: break

            # 광고 제외
            if item.select_one(".pal-c-ProductCardFooter__sponsored"):
                continue

            try:
                # 상품 정보 파싱
                brand = item.select_one(".pal-c-ProductCardBody--brandName p").get_text(strip=True)
                title_link = item.select_one("a.pal-c-Link")
                title = title_link.get_text(strip=True)
                product_url = title_link.get("href")
                if not product_url.startswith("http"):
                    product_url = "https://www.ulta.com" + product_url

                # URL에서 ID 추출
                product_id = product_url.split("-")[-1].split("?")[0]

                rank_data_list.append({
                    "rank": rank_count,
                    "brand": brand,
                    "title": title,
                    "url": product_url,
                    "product_id": product_id,
                    "platform": "Ulta",
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                })
                print(f"\n📍 {rank_count}위: [{brand}] {title[:30]}")

                # ── 리뷰 수집 실행 ──
                reviews = get_product_reviews_2years(product_id, title)
                all_review_master.extend(reviews)

                rank_count += 1

            except Exception as e:
                print(f"⚠️ 파싱 오류: {e}")
                continue

        # JSONL 저장
        with open(RANK_SAVE_FILE, "w", encoding="utf-8") as f:
            for entry in rank_data_list: f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        with open(REVIEW_SAVE_FILE, "w", encoding="utf-8") as f:
            for review in all_review_master: f.write(json.dumps(review, ensure_ascii=False) + "\n")

        print(f"\n📊 작업 완료: 상품 {len(rank_data_list)}개 / 총 리뷰 {len(all_review_master)}개 저장됨")

    finally:
        driver.quit()

if __name__ == "__main__":
    fetch_ulta_data()

📡 Ulta 접속 중 (기준일: 2024-03-20)

📍 1위: [IT Cosmetics] IT Cosmetics Do It All Sheer T

  🚀 [IT Cosmetics Do It All Sheer T] 리뷰 수집 (기준: 2024-03-20 이후)
    🔄 수집 중... (누적: 3201개)
    ✨ 3201개 수집 완료

📍 2위: [medicube] medicube Zero Pore Pad

  🚀 [medicube Zero Pore Pad] 리뷰 수집 (기준: 2024-03-20 이후)
    🔄 수집 중... (누적: 127개)
    ✨ 127개 수집 완료

📍 3위: [The Ordinary] The Ordinary Glycolic Acid 7% 

  🚀 [The Ordinary Glycolic Acid 7% ] 리뷰 수집 (기준: 2024-03-20 이후)
    📍 2년 이전 리뷰 도달 (2024-03-21 06:48) - 수집 종료

    ✨ 463개 수집 완료

📍 4위: [Clinique] Clinique Moisture Surge 100H A

  🚀 [Clinique Moisture Surge 100H A] 리뷰 수집 (기준: 2024-03-20 이후)
    📍 2년 이전 리뷰 도달 (2024-03-22 03:01) - 수집 종료

    ✨ 507개 수집 완료

📍 5위: [MAËLYS] MAËLYS GET-DREAMY Overnight To

  🚀 [MAËLYS GET-DREAMY Overnight To] 리뷰 수집 (기준: 2024-03-20 이후)
    📍 2년 이전 리뷰 도달 (2024-03-21 09:00) - 수집 종료

    ✨ 4535개 수집 완료

📍 6위: [Clinique] Clinique Even Better Makeup Br

  🚀 [Clinique Even Better Makeup Br] 리뷰 수집 (기준: 2024-03-20 이후)
    📍 2년 이전 리뷰 도달 (2024-03

# ulta 번역코드

In [2]:
import json
import time
import re
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from deep_translator import GoogleTranslator

# ── 설정 ──────────────────────────────────────────────────────────
INPUT_FILE  = './ulta_reviews_master.jsonl'
OUTPUT_FILE = 'ulta_master_translated_en_ko.jsonl'

# 울타 데이터의 키 매칭
BODY_COL    = 'comment'   # 리뷰 본문
HEAD_COL    = 'headline'  # 리뷰 제목
ITEM_ID_COL = 'product_id'
RATING_COL  = 'rating'

# [병렬 처리 설정]
CHUNK_SIZE  = 5      # 묶음 번역 단위
MAX_WORKERS = 4      # 영어-한국어는 차단 위험이 낮아 4 유지 가능
MAX_RETRIES = 3      
RETRY_SLEEP = 2.0    
CHUNK_DELAY = 0.3    
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_numbered(texts: list) -> str:
    return "\n".join(f"[{i+1}] {str(t).strip()}" for i, t in enumerate(texts))

def parse_numbered(text: str, expected_n: int) -> list:
    pattern = re.compile(r'\[(\d+)\]\s*(.*?)(?=\[\d+\]|$)', re.DOTALL)
    found = pattern.findall(text)
    result = {int(idx): body.strip() for idx, body in found}
    return [result.get(i + 1, "") for i in range(expected_n)]

def translate_single(text: str, src: str, tgt: str) -> str:
    if not text or not str(text).strip(): return ""
    for attempt in range(MAX_RETRIES):
        try:
            res = GoogleTranslator(source=src, target=tgt).translate(str(text))
            if res: return res.strip()
        except:
            time.sleep(RETRY_SLEEP * (attempt + 1))
    return "번역실패"

def process_chunk(texts: list, src: str, tgt: str) -> list:
    if not texts: return []
    joined = build_numbered(texts)
    for attempt in range(MAX_RETRIES):
        try:
            translated = GoogleTranslator(source=src, target=tgt).translate(joined)
            if not translated: raise ValueError("응답 없음")
            parts = parse_numbered(translated, len(texts))
            if all(p.strip() for p in parts): return parts
            for i, p in enumerate(parts):
                if not p: parts[i] = translate_single(texts[i], src, tgt)
            return parts
        except:
            time.sleep(RETRY_SLEEP * (attempt + 1))
    return [translate_single(t, src, tgt) for t in texts]

def translate_workflow(texts: list):
    """en -> ko 단일 단계 번역"""
    ko_texts = process_chunk(texts, 'en', 'ko')
    time.sleep(CHUNK_DELAY)
    return ko_texts

def main():
    print(f"📥 Ulta 데이터 로딩 중: {INPUT_FILE}")
    records = []
    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip(): records.append(json.loads(line))
    except FileNotFoundError:
        print("❌ 입력 파일을 찾을 수 없습니다.")
        return

    df = pd.DataFrame(records)
    total_len = len(df)
    print(f"✅ 총 {total_len:,}건 확인")

    # 1. 리뷰 제목(headline) 번역
    print(f"\n🏷 1/2 단계: 리뷰 제목 번역 시작...")
    heads = df[HEAD_COL].fillna("").astype(str).tolist()
    head_chunks = [heads[i:i + CHUNK_SIZE] for i in range(0, total_len, CHUNK_SIZE)]
    
    all_head_ko = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        results = list(tqdm(executor.map(translate_workflow, head_chunks), total=len(head_chunks), desc="제목 번역 중"))
        for chunk in results: all_head_ko.extend(chunk)
    df['headline_ko'] = all_head_ko[:total_len]

    # 2. 리뷰 본문(comment) 번역
    print(f"\n📝 2/2 단계: 리뷰 본문 번역 시작...")
    bodies = df[BODY_COL].fillna("").astype(str).tolist()
    body_chunks = [bodies[i:i + CHUNK_SIZE] for i in range(0, total_len, CHUNK_SIZE)]
    
    all_body_ko = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        results = list(tqdm(executor.map(translate_workflow, body_chunks), total=len(body_chunks), desc="본문 번역 중"))
        for chunk in results: all_body_ko.extend(chunk)
    df['comment_ko'] = all_body_ko[:total_len]

    # 💾 결과 저장
    print(f"\n💾 결과 저장 중: {OUTPUT_FILE}")
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for record in df.to_dict(orient='records'):
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

    print(f"✨ 모든 작업 완료! ({OUTPUT_FILE})")

if __name__ == "__main__":
    main()

📥 Ulta 데이터 로딩 중: ./ulta_reviews_master.jsonl
✅ 총 10,889건 확인

🏷 1/2 단계: 리뷰 제목 번역 시작...


제목 번역 중: 100%|██████████| 2178/2178 [12:50<00:00,  2.83it/s]



📝 2/2 단계: 리뷰 본문 번역 시작...


본문 번역 중: 100%|██████████| 2178/2178 [14:35<00:00,  2.49it/s] 


💾 결과 저장 중: ulta_master_translated_en_ko.jsonl
✨ 모든 작업 완료! (ulta_master_translated_en_ko.jsonl)


# ulta kebert 1차 분류

In [1]:
import re
import json
import pandas as pd
from tqdm import tqdm
from keybert import KeyBERT

# =========================
# 설정
# =========================
INPUT_FILE   = "./ulta_master_translated_en_ko.jsonl"
OUTPUT_CSV   = "./ulta_keybert_en_categorized.csv"
OUTPUT_JSONL = "./ulta_keybert_en_categorized.jsonl"

TEXT_COL       = "comment"
TOP_N_KEYWORDS = 5

CATEGORY_KEYWORDS = {
    "효과_성분": [
        "moistur", "hydrat", "hydrating", "whitening", "brighten", "elastic", "firm",
        "wrinkle", "anti-aging", "pore", "glow", "radian", "regenerat",
        "sooth", "calm", "antioxidant", "absorb", "penetrat", "tone", "improve",
        "vitamin", "retinol", "hyaluronic", "ceramide", "niacinamide", "peptid", "vegan",
        "plump", "clear", "even", "spot", "pigment", "aha", "bha", "acid", "exfoliat",
        # 추가: 피부타입 맥락
        "oily skin", "combination skin", "sensitive skin", "works for my skin",
        "dry skin type", "acne-prone skin",
        # 추가: 결과/시간 표현
        "result", "noticeabl", "overnight", "immediately", "after using",
        "after one week", "after a month", "after two week",
        # 추가: 사용 맥락
        "routine", "layer", "morning", "night cream",
        # 추가: 선케어 (제품군 해당 시)
        "spf", "sunscreen", "uv", "sun protect", "reef safe",
    ],
    "사용감_텍스처": [
        "appl", "blend", "texture", "consistenc", "watery", "runny", "thick", "viscos",
        "stick", "tacky", "fresh", "light", "weightless", "heavy", "soft", "smooth",
        "stiff", "greasy", "pill", "flake", "feel", "finish", "rub",
        "oily", "matte", "dewy", "sink", "patchy", "chalky", "white cast",
        # 추가: 메이크업 베이스/선케어 관련
        "pore-filling", "pore filling", "blur", "setting", "blot", "primer",
        "spread", "glide", "pack", "apply thin", "build up",
    ],
    "향_냄새": [
        "scent", "smell", "fragranc", "unscented", "fragrance-free", "odor",
        "subtle", "mild", "strong", "overpowering", "artificial", "natural", "perfume",
        "stink", "aroma", "nose",
        # 추가
        "whiff", "chemical smell", "medicin", "floral", "citrus",
    ],
    "피부_트러블_부작용": [
        "trouble", "breakout", "pimple", "acne", "irritat", "sting", "burn", "itch",
        "red", "redness", "peel", "tight", "sensitiv", "allerg", "dermatitis",
        "reaction", "side effect", "break out", "rash", "harsh",
        "drying", "dried out", "flaky", "dry patch",
        "clog", "purg", "cyst", "bump",
        # 추가: 자극 표현
        "tingle", "sting", "inflam", "swell", "hive", "welt",
        "made my skin worse", "broke me out", "not agree",
    ],
    "포장_배송": [
        "packag", "box", "bottle", "container", "case", "pump", "tube", "ship",
        "deliver", "late", "slow", "arriv", "damag", "broken",
        "leak", "spill", "wrap",
        "fast ship", "arrived fast", "quick deliver",
        "dropper", "cap", "lid", "spray", "nozzle", "shipped", "unseal",
        # 추가
        "packaging", "travel size", "full size", "well-packaged", "poorly packaged",
        "dent", "crush", "tamper",
    ],
    "가격_가성비": [
        "price", "cost", "valu", "expensiv", "pricy", "cheap", "afford", "reasonabl",
        "sale", "discount", "coupon", "buck", "money", "worth", "deal", "size", "amount",
        "pricey", "bargain", "rip off", "waste",
        # 추가: 용량 관련 가성비 표현
        "goes a long way", "a little goes", "last a long time", "last me",
        "small amount", "tiny bit", "lasts forever", "run out fast", "finish quickly",
    ],
    "고객서비스": [
        "custom", "service", "support", "respond", "response", "refund", "return",
        "exchang", "complain", "inquir", "answer", "contact", "issue",
        # 추가
        "seller", "vendor", "representative", "chat", "email them", "called",
        "waited", "resolve", "compensat",
    ],
    "제품불량": [
        "defect", "defective", "faulty", "bug", "dirt", "contaminat", "spoil",
        "weird", "fake", "counterfeit", "knockoff", "differ", "mold", "trash",
        "expir", "rancid", "separat", "empty",
        # 추가
        "smell off", "color off", "look different", "wrong product", "not what",
        "different from", "old stock", "bad batch",
    ],
    "재구매_추천": [
        "repurchas", "buy again", "reorder", "recommend", "holy grail", "staple",
        "go-to", "favorit", "gift", "friend", "keep us", "definitely",
        "love it",
        "hg", "restock", "10/10", "must have", "obsessed",
        # 추가: 비교/전환 표현
        "better than", "switch from", "switch to", "compared to", "used to use",
        "replace", "converted",
        # 추가: 일반 강한 긍정 (내용 없는 짧은 리뷰 흡수)
        "highly recommend", "great product", "works well", "works great",
        "amazing product", "excellent", "perfect product", "love this",
        "love how", "love that", "so good", "so happy",
    ],
    
     "부정_리뷰": [
        # 추가: 부정 추천 표현
        "disappoint", "not recommend", "waste of money", "regret buying",
    ],

    "커버력_색상": [
        "cover", "coverage", "color", "shade", "tone", "tint", "pigment",
        "bright", "dark", "ashy", "oxidiz", "orang", "yellow", "match", "pale",
        "undertone", "fair", "sheer", "opaque", "swatch",
        # 추가
        "full coverage", "medium coverage", "buildable", "natural finish",
        "foundation", "concealer", "bb cream", "cc cream", "tinted",
        "skin tone", "complexion", "too light", "too dark", "perfect match",
    ],
    "지속력_밀착력": [
        "last", "lasting", "longevity", "stay", "adher", "crease", "melt", "fade",
        "long-lasting", "all day", "wear", "hold", "slip", "smudg", "transfer",
        "rub off", "budge", "separate",
        # 추가: 환경 내구성
        "sweat", "sweatproof", "waterproof", "water resistant", "humid",
        "through the day", "by noon", "by midday", "hours later",
        "8 hour", "12 hour", "24 hour",
    ],
}

# =========================
# 유틸
# =========================
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

def clean_light(text):
    text = str(text).replace("\n", " ").replace("\r", " ")
    return re.sub(r"\s+", " ", text).strip()

def extract_keywords(text, top_n=5):
    text = str(text).strip()
    if not text:
        return []
    try:
        kws = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 2),
            stop_words="english",
            top_n=top_n,
            use_mmr=True,
            diversity=0.5
        )
        return [kw for kw, _ in kws]
    except Exception:
        return []

def rule_classify(keywords: list, text: str) -> tuple:
    combined = " ".join(keywords).lower() + " " + text.lower()
    scores = {}
    for cat, kw_list in CATEGORY_KEYWORDS.items():
        count = sum(1 for kw in kw_list if kw in combined)
        if count > 0:
            scores[cat] = count

    if not scores:
        return "unclassified", ["unclassified"]

    sorted_cats = sorted(scores, key=scores.get, reverse=True)
    return sorted_cats[0], sorted_cats[:3]

# =========================
# 모델 로드
# =========================
kw_model = KeyBERT("all-MiniLM-L6-v2")

# =========================
# 데이터 로드
# =========================
df = load_jsonl(INPUT_FILE)
df = df.dropna(subset=[TEXT_COL]).copy()
df = df[df[TEXT_COL].astype(str).str.strip() != ""].reset_index(drop=True)
df["text_for_model"] = df[TEXT_COL].apply(clean_light)
print(f"유효 데이터: {len(df):,}건")

# =========================
# KeyBERT 키워드 추출 + 규칙 분류
# =========================
keywords_list      = []
primary_categories = []
categories_list    = []

for text in tqdm(df["text_for_model"], desc="KeyBERT 분류 (EN)"):
    kws           = extract_keywords(text, top_n=TOP_N_KEYWORDS)
    primary, cats = rule_classify(kws, text)
    keywords_list.append(kws)
    primary_categories.append(primary)
    categories_list.append(cats)

df["keybert_keywords"]  = keywords_list
df["primary_category"]  = primary_categories
df["categories"]        = categories_list

# =========================
# 결과 출력
# =========================
total        = len(df)
classified   = (df["primary_category"] != "unclassified").sum()
unclassified = (df["primary_category"] == "unclassified").sum()

print(f"\n분류 완료: {classified:,}건 ({classified/total*100:.1f}%)")
print(f"미분류:    {unclassified:,}건 ({unclassified/total*100:.1f}%)")
print("\n카테고리 분포:")
print(df["primary_category"].value_counts())

# =========================
# 저장
# =========================
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

print(f"\nCSV:   {OUTPUT_CSV}")
print(f"JSONL: {OUTPUT_JSONL}")

print("\n샘플:")
print(df[[TEXT_COL, "keybert_keywords", "primary_category", "categories"]].head(10))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


유효 데이터: 10,889건


KeyBERT 분류 (EN):   1%|          | 75/10889 [00:02<05:30, 32.77it/s]


KeyboardInterrupt: 

# ulta gpt 2차 분류

In [ ]:
import json
import re
from openai import OpenAI

# ── 설정 ──────────────────────────────────────────────────────────────
INPUT_JSONL  = "./ulta_keybert_en_categorized.jsonl"  # KeyBERT 결과 JSONL
OUTPUT_JSONL = "./ulta_final_categorized.jsonl"
BATCH_SIZE   = 30
TEXT_COL     = "comment"

CATEGORIES = [
    "효과_성분", "사용감_텍스처", "향_냄새", "피부_트러블_부작용","부정_리뷰",
    "포장_배송", "가격_가성비", "고객서비스", "제품불량",
    "재구매_추천", "커버력_색상", "지속력_밀착력", "미분류"
]

client = OpenAI()

# ── GPT 배치 분류 ──────────────────────────────────────────────────────
def gpt_classify_batch(batch: list[dict]) -> dict:
    """batch: [{"idx": i, "keywords": [...], "text": "..."}]
    반환: {idx: {"primary_category": ..., "categories": [...]}}
    """
    prompt_items = "\n".join(
        f"[{item['idx']}] keywords={item['keywords']} | text={item['text'][:300]}"
        for item in batch
    )
    system_msg = f"""뷰티 제품 리뷰를 아래 카테고리 중 하나로 분류하세요.
카테고리: {CATEGORIES}

각 리뷰에 대해 JSON 배열로 응답하세요:
[{{"idx": 번호, "primary_category": "카테고리명", "categories": ["카테고리1", ...]}}]
primary_category는 가장 핵심 카테고리 1개, categories는 해당되는 카테고리 모두."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": prompt_items}
        ],
        temperature=0
    )

    raw = response.choices[0].message.content
    # JSON 배열 파싱
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    if not match:
        return {}
    results = json.loads(match.group())
    return {r["idx"]: r for r in results}

# ── 데이터 로드 ────────────────────────────────────────────────────────
records = []
with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

classified   = [r for r in records if r.get("primary_category") != "unclassified"]
unclassified = [r for r in records if r.get("primary_category") == "unclassified"]

print(f"전체: {len(records)} | 분류완료: {len(classified)} | GPT 재분류 대상: {len(unclassified)}")

# ── GPT 배치 실행 ──────────────────────────────────────────────────────
idx_to_record = {i: rec for i, rec in enumerate(unclassified)}
batches = [
    [
        {
            "idx": i,
            "keywords": rec.get("keybert_keywords", []),
            "text": str(rec.get(TEXT_COL, ""))[:300]
        }
        for i, rec in list(idx_to_record.items())[start:start+BATCH_SIZE]
    ]
    for start in range(0, len(unclassified), BATCH_SIZE)
]

print(f"배치 수: {len(batches)} ({BATCH_SIZE}건씩)")

for b_idx, batch in enumerate(batches):
    results = gpt_classify_batch(batch)
    for item in batch:
        i = item["idx"]
        if i in results:
            idx_to_record[i]["primary_category"] = results[i]["primary_category"]
            idx_to_record[i]["categories"]       = results[i]["categories"]
        else:
            idx_to_record[i]["primary_category"] = "이분류"
            idx_to_record[i]["categories"]       = []
    print(f"  배치 {b_idx+1}/{len(batches)} 완료")

# ── 결과 저장 ──────────────────────────────────────────────────────────
final_records = classified + list(idx_to_record.values())

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for rec in final_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"\n저장 완료: {OUTPUT_JSONL} ({len(final_records)}건)")

# ── 분류 결과 확인 ─────────────────────────────────────────────────────
from collections import Counter
cats = [r["primary_category"] for r in final_records]
for cat, cnt in Counter(cats).most_common():
    print(f"  {cat}: {cnt}")

## 효과_성분 세분화 KeyBERT 재분류 + GPT 2차 분류

In [ ]:
import re
import json
import pandas as pd
from tqdm import tqdm
from keybert import KeyBERT
from openai import OpenAI
from collections import Counter

INPUT_FILE   = "./ulta_master_translated_en_ko.jsonl"
OUTPUT_JSONL = "./ulta_final_categorized_v2.jsonl"
TEXT_COL     = "Review_en"
BATCH_SIZE   = 30

CATEGORY_KEYWORDS = {
    "주름_노화": [
        "wrinkle", "anti-aging", "antiaging", "fine line", "age spot", "sagging",
        "retinol", "peptid", "adenosine", "bakuchiol", "firm", "elastic", "lifting",
        "tighten", "regenerat",
    ],
    "보습_수분": [
        "moistur", "hydrat", "hydrating", "plump", "dehydrat", "quench", "dewiness",
        "hyaluronic", "ceramide", "glycerin", "water",
    ],
    "미백_브라이트닝": [
        "whitening", "brighten", "bright", "glow", "radian", "tone", "even", "clear",
        "spot", "pigment", "discolor", "dull",
        "niacinamide", "vitamin c", "arbutin", "tranexamic",
    ],
    "진정_장벽": [
        "sooth", "calm", "barrier", "sensitiv",
        "centella", "cica", "madecassoside", "panthenol", "allantoin", "madeca",
    ],
    "모공_각질": [
        "pore", "exfoliat", "aha", "bha", "acid", "peel", "blackhead",
        "sebum", "rough", "bumpy",
    ],
    "사용감_텍스처": [
        "appl", "blend", "texture", "consistenc", "watery", "runny", "thick", "viscos",
        "stick", "tacky", "fresh", "light", "weightless", "heavy", "soft", "smooth",
        "stiff", "greasy", "pill", "flake", "feel", "finish", "rub",
        "oily", "matte", "dewy", "sink", "patchy", "chalky", "white cast",
        "pore-filling", "blur", "setting", "blot", "primer", "spread", "glide",
    ],
    "향_냄새": [
        "scent", "smell", "fragranc", "unscented", "fragrance-free", "odor",
        "subtle", "mild", "strong", "overpowering", "artificial", "natural", "perfume",
        "stink", "aroma", "nose", "whiff", "chemical smell", "floral", "citrus",
    ],
    "피부_트러블_부작용": [
        "trouble", "breakout", "pimple", "acne", "irritat", "sting", "burn", "itch",
        "red", "redness", "tight", "allerg", "dermatitis", "reaction", "side effect",
        "break out", "rash", "harsh", "drying", "dried out", "flaky", "dry patch",
        "clog", "purg", "cyst", "bump", "tingle", "inflam", "swell",
        "made my skin worse", "broke me out",
    ],
    "포장_배송": [
        "packag", "box", "bottle", "container", "case", "pump", "tube", "ship",
        "deliver", "late", "slow", "arriv", "damag", "broken", "leak", "spill",
        "dropper", "cap", "lid", "spray", "nozzle", "dent", "crush",
    ],
    "가격_가성비": [
        "price", "cost", "valu", "expensiv", "pricy", "cheap", "afford", "reasonabl",
        "sale", "discount", "money", "worth", "deal", "pricey", "bargain",
        "goes a long way", "a little goes", "lasts forever",
    ],
    "재구매_추천": [
        "repurchas", "buy again", "reorder", "recommend", "holy grail", "staple",
        "favorit", "must have", "obsessed", "better than", "switch from",
        "highly recommend", "great product", "works well", "love this", "so good",
    ],
    "부정_리뷰": [
        "disappoint", "not recommend", "waste of money", "regret buying",
    ],
    "지속력_밀착력": [
        "last", "lasting", "longevity", "stay", "adher", "crease", "melt", "fade",
        "long-lasting", "all day", "wear", "hold", "smudg", "transfer", "rub off",
        "sweatproof", "waterproof", "water resistant",
    ],
    "고객서비스": [
        "custom", "service", "support", "respond", "refund", "return", "exchang",
        "complain", "answer", "contact", "seller", "vendor",
    ],
    "제품불량": [
        "defect", "defective", "faulty", "contaminat", "spoil", "fake", "counterfeit",
        "expir", "rancid", "wrong product", "different from", "bad batch",
    ],
}

CATEGORIES = list(CATEGORY_KEYWORDS.keys()) + ["미분류"]

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

def clean_light(text):
    text = str(text).replace("\n", " ").replace("\r", " ")
    return re.sub(r"\s+", " ", text).strip()

def extract_keywords_func(text, model, top_n=5):
    text = str(text).strip()
    if not text:
        return []
    try:
        kws = model.extract_keywords(
            text, keyphrase_ngram_range=(1, 2),
            stop_words="english", top_n=top_n,
            use_mmr=True, diversity=0.5
        )
        return [kw for kw, _ in kws]
    except Exception:
        return []

def rule_classify(keywords, text):
    combined = " ".join(keywords).lower() + " " + text.lower()
    scores = {}
    for cat, kw_list in CATEGORY_KEYWORDS.items():
        count = sum(1 for kw in kw_list if kw in combined)
        if count > 0:
            scores[cat] = count
    if not scores:
        return "unclassified", ["unclassified"]
    sorted_cats = sorted(scores, key=scores.get, reverse=True)
    return sorted_cats[0], sorted_cats[:3]

print("🚀 KeyBERT 모델 로딩 중...")
kw_model = KeyBERT("all-MiniLM-L6-v2")

df = load_jsonl(INPUT_FILE)
df = df.dropna(subset=[TEXT_COL]).copy()
df = df[df[TEXT_COL].astype(str).str.strip() != ""].reset_index(drop=True)
df["text_for_model"] = df[TEXT_COL].apply(clean_light)
print(f"📊 유효 데이터: {len(df):,}건")

keywords_list, primary_categories, categories_list = [], [], []
for text in tqdm(df["text_for_model"], desc="KeyBERT 1차 분류"):
    kws = extract_keywords_func(text, kw_model)
    primary, cats = rule_classify(kws, text)
    keywords_list.append(kws)
    primary_categories.append(primary)
    categories_list.append(cats)

df["keybert_keywords"] = keywords_list
df["primary_category"] = primary_categories
df["categories"]       = categories_list

classified_df   = df[df["primary_category"] != "unclassified"]
unclassified_df = df[df["primary_category"] == "unclassified"]
print(f"\n✅ 1차 분류 완료: {len(classified_df):,}건 | GPT 대상: {len(unclassified_df):,}건")
print(classified_df["primary_category"].value_counts())

client = OpenAI()

def gpt_classify_batch(batch):
    prompt_items = "\n".join(
        f"[{item['idx']}] keywords={item['keywords']} | text={item['text'][:300]}"
        for item in batch
    )
    system_msg = f"""뷰티 제품 리뷰를 아래 카테고리 중 하나로 분류하세요.
카테고리: {CATEGORIES}
각 리뷰에 대해 JSON 배열로 응답하세요:
[{{"idx": 번호, "primary_category": "카테고리명", "categories": ["카테고리1", ...]}}]
primary_category는 가장 핵심 카테고리 1개, categories는 해당되는 카테고리 모두."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": system_msg}, {"role": "user", "content": prompt_items}],
        temperature=0,
    )
    raw = response.choices[0].message.content
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    if not match:
        return {}
    return {r["idx"]: r for r in json.loads(match.group())}

records              = df.to_dict("records")
classified_records   = [r for r in records if r["primary_category"] != "unclassified"]
unclassified_records = [r for r in records if r["primary_category"] == "unclassified"]
idx_to_record = {i: rec for i, rec in enumerate(unclassified_records)}
batches = [
    [{"idx": i, "keywords": rec.get("keybert_keywords", []), "text": str(rec.get(TEXT_COL, ""))[:300]}
     for i, rec in list(idx_to_record.items())[s:s + BATCH_SIZE]]
    for s in range(0, len(unclassified_records), BATCH_SIZE)
]

print(f"\n🤖 GPT 2차 분류 시작: {len(batches)}배치")
for b_idx, batch in enumerate(batches):
    results = gpt_classify_batch(batch)
    for item in batch:
        i = item["idx"]
        if i in results:
            idx_to_record[i]["primary_category"] = results[i]["primary_category"]
            idx_to_record[i]["categories"]       = results[i]["categories"]
        else:
            idx_to_record[i]["primary_category"] = "미분류"
            idx_to_record[i]["categories"]       = []
    print(f"  배치 {b_idx+1}/{len(batches)} 완료")

final_records = classified_records + list(idx_to_record.values())
with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for rec in final_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"\n💾 저장 완료: {OUTPUT_JSONL} ({len(final_records):,}건)")
print("\n[최종 카테고리 분포]")
for cat, cnt in Counter(r["primary_category"] for r in final_records).most_common():
    print(f"  {cat}: {cnt:,}")
